In [0]:
# Define Bronze and Silver data paths

bronze_path = "/Volumes/workspace/default/t_series_forecasting_data/bronze_data"
silver_path = "/Volumes/workspace/default/t_series_forecasting_data/silver_data"

In [0]:
# Read all Bronze Delta tables

df_errors = spark.read.format("delta").load(f"{bronze_path}/PdM_errors")
df_failures = spark.read.format("delta").load(f"{bronze_path}/PdM_failures")
df_machines = spark.read.format("delta").load(f"{bronze_path}/PdM_machines")
df_maint = spark.read.format("delta").load(f"{bronze_path}/PdM_maint")
df_telemetry = spark.read.format("delta").load(f"{bronze_path}/PdM_telemetry")

In [0]:
# Check Silver input row counts from Bronze tables

print("Errors:", df_errors.count())
print("Failures:", df_failures.count())
print("Machines:", df_machines.count())
print("Maintenance:", df_maint.count())
print("Telemetry:", df_telemetry.count())

Errors: 3919
Failures: 761
Machines: 100
Maintenance: 3286
Telemetry: 876100


In [0]:
# Check null values in all Bronze tables

from pyspark.sql import functions as F

for name, df in [
    ("PdM_errors", df_errors),
    ("PdM_failures", df_failures),
    ("PdM_machines", df_machines),
    ("PdM_maint", df_maint),
    ("PdM_telemetry", df_telemetry)
]:
    print(f"\n===== {name} =====")
    df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]).show()


===== PdM_errors =====
+--------+---------+-------+
|datetime|machineID|errorID|
+--------+---------+-------+
|       0|        0|      0|
+--------+---------+-------+


===== PdM_failures =====
+--------+---------+-------+
|datetime|machineID|failure|
+--------+---------+-------+
|       0|        0|      0|
+--------+---------+-------+


===== PdM_machines =====
+---------+-----+---+
|machineID|model|age|
+---------+-----+---+
|        0|    0|  0|
+---------+-----+---+


===== PdM_maint =====
+--------+---------+----+
|datetime|machineID|comp|
+--------+---------+----+
|       0|        0|   0|
+--------+---------+----+


===== PdM_telemetry =====
+--------+---------+----+------+--------+---------+
|datetime|machineID|volt|rotate|pressure|vibration|
+--------+---------+----+------+--------+---------+
|       0|        0|   0|     0|       0|        0|
+--------+---------+----+------+--------+---------+



In [0]:
# Check duplicate rows in all Bronze tables

for name, df in [
    ("PdM_errors", df_errors),
    ("PdM_failures", df_failures),
    ("PdM_machines", df_machines),
    ("PdM_maint", df_maint),
    ("PdM_telemetry", df_telemetry)
]:
    total_rows = df.count()
    unique_rows = df.dropDuplicates().count()
    duplicates = total_rows - unique_rows
    
    print(f"{name}: Total = {total_rows}, Duplicates = {duplicates}")

PdM_errors: Total = 3919, Duplicates = 0
PdM_failures: Total = 761, Duplicates = 0
PdM_machines: Total = 100, Duplicates = 0
PdM_maint: Total = 3286, Duplicates = 0
PdM_telemetry: Total = 876100, Duplicates = 0


In [0]:
# Check the available time range in event tables

for name, df in [
    ("PdM_errors", df_errors),
    ("PdM_failures", df_failures),
    ("PdM_maint", df_maint),
    ("PdM_telemetry", df_telemetry)
]:
    print(f"\n===== {name} =====")
    df.select(
        F.min("datetime").alias("start_time"),
        F.max("datetime").alias("end_time")
    ).show()


===== PdM_errors =====
+-------------------+-------------------+
|         start_time|           end_time|
+-------------------+-------------------+
|2015-01-01 06:00:00|2016-01-01 05:00:00|
+-------------------+-------------------+


===== PdM_failures =====
+-------------------+-------------------+
|         start_time|           end_time|
+-------------------+-------------------+
|2015-01-02 03:00:00|2015-12-31 06:00:00|
+-------------------+-------------------+


===== PdM_maint =====
+-------------------+-------------------+
|         start_time|           end_time|
+-------------------+-------------------+
|2014-06-01 06:00:00|2016-01-01 06:00:00|
+-------------------+-------------------+


===== PdM_telemetry =====
+-------------------+-------------------+
|         start_time|           end_time|
+-------------------+-------------------+
|2015-01-01 06:00:00|2016-01-01 06:00:00|
+-------------------+-------------------+



In [0]:
# Check the maintenance data time range

df_maint.select(
    F.min("datetime").alias("start_time"),
    F.max("datetime").alias("end_time")
).show()

+-------------------+-------------------+
|         start_time|           end_time|
+-------------------+-------------------+
|2014-06-01 06:00:00|2016-01-01 06:00:00|
+-------------------+-------------------+



In [0]:
# Check null values in errors and failures tables

print("===== PdM_errors =====")
df_errors.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_errors.columns
]).show()

print("===== PdM_failures =====")
df_failures.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_failures.columns
]).show()

===== PdM_errors =====
+--------+---------+-------+
|datetime|machineID|errorID|
+--------+---------+-------+
|       0|        0|      0|
+--------+---------+-------+

===== PdM_failures =====
+--------+---------+-------+
|datetime|machineID|failure|
+--------+---------+-------+
|       0|        0|      0|
+--------+---------+-------+



In [0]:
# Display records from each table

print("===== PdM_errors =====")
display(df_errors.limit(20))

print("===== PdM_failures =====")
display(df_failures.limit(20))

print("===== PdM_machines =====")
display(df_machines.limit(20))

print("===== PdM_maint =====")
display(df_maint.limit(20))

print("===== PdM_telemetry =====")
display(df_telemetry.limit(20))

===== PdM_errors =====


datetime,machineID,errorID
2015-01-03T07:00:00.000Z,1,error1
2015-01-03T20:00:00.000Z,1,error3
2015-01-04T06:00:00.000Z,1,error5
2015-01-10T15:00:00.000Z,1,error4
2015-01-22T10:00:00.000Z,1,error4
2015-01-25T15:00:00.000Z,1,error4
2015-01-27T04:00:00.000Z,1,error1
2015-03-03T22:00:00.000Z,1,error2
2015-03-05T06:00:00.000Z,1,error1
2015-03-20T18:00:00.000Z,1,error1


===== PdM_failures =====


datetime,machineID,failure
2015-01-05T06:00:00.000Z,1,comp4
2015-03-06T06:00:00.000Z,1,comp1
2015-04-20T06:00:00.000Z,1,comp2
2015-06-19T06:00:00.000Z,1,comp4
2015-09-02T06:00:00.000Z,1,comp4
2015-10-17T06:00:00.000Z,1,comp2
2015-12-16T06:00:00.000Z,1,comp4
2015-03-19T06:00:00.000Z,2,comp1
2015-03-19T06:00:00.000Z,2,comp2
2015-04-18T06:00:00.000Z,2,comp2


===== PdM_machines =====


machineID,model,age
1,model3,18
2,model4,7
3,model3,8
4,model3,7
5,model3,2
6,model3,7
7,model3,20
8,model3,16
9,model4,7
10,model3,10


===== PdM_maint =====


datetime,machineID,comp
2014-06-01T06:00:00.000Z,1,comp2
2014-07-16T06:00:00.000Z,1,comp4
2014-07-31T06:00:00.000Z,1,comp3
2014-12-13T06:00:00.000Z,1,comp1
2015-01-05T06:00:00.000Z,1,comp4
2015-01-05T06:00:00.000Z,1,comp1
2015-01-20T06:00:00.000Z,1,comp3
2015-01-20T06:00:00.000Z,1,comp1
2015-02-04T06:00:00.000Z,1,comp4
2015-02-04T06:00:00.000Z,1,comp3


===== PdM_telemetry =====


datetime,machineID,volt,rotate,pressure,vibration
2015-01-01T06:00:00.000Z,1,176.217853015625,418.504078221616,113.077935462083,45.0876857639276
2015-01-01T07:00:00.000Z,1,162.87922289706,402.747489565395,95.4605253823187,43.4139726834815
2015-01-01T08:00:00.000Z,1,170.989902405567,527.349825452291,75.2379048586662,34.1788471214451
2015-01-01T09:00:00.000Z,1,162.462833264092,346.149335043074,109.248561276504,41.1221440884256
2015-01-01T10:00:00.000Z,1,157.61002119306,435.376873016938,111.886648210168,25.9905109982024
2015-01-01T11:00:00.000Z,1,172.504839196295,430.323362106675,95.9270416939636,35.6550173268837
2015-01-01T12:00:00.000Z,1,156.556030606329,499.071623068962,111.755684290096,42.7539196974773
2015-01-01T13:00:00.000Z,1,172.522780814836,409.624717000438,101.00108276407,35.4820086610704
2015-01-01T14:00:00.000Z,1,175.324523915223,398.648780707752,110.624360548654,45.4822868466294
2015-01-01T15:00:00.000Z,1,169.218423246933,460.850669930244,104.848229967003,39.9017354356787


In [0]:
# Create clean Silver DataFrames

silver_errors = df_errors.dropDuplicates()
silver_failures = df_failures.dropDuplicates()
silver_machines = df_machines.dropDuplicates()
silver_maint = df_maint.dropDuplicates()
silver_telemetry = df_telemetry.dropDuplicates()

In [0]:
# Store cleaned errors data as a Silver Delta table
silver_errors.write.format("delta").save(f"{silver_path}/PdM_errors")

# Store cleaned failures data as a Silver Delta table
silver_failures.write.format("delta").save(f"{silver_path}/PdM_failures")

# Store cleaned machine data as a Silver Delta table
silver_machines.write.format("delta").save(f"{silver_path}/PdM_machines")

# Store cleaned maintenance data as a Silver Delta table
silver_maint.write.format("delta").save(f"{silver_path}/PdM_maint")

# Store cleaned telemetry data as a Silver Delta table
silver_telemetry.write.format("delta").save(f"{silver_path}/PdM_telemetry")

In [0]:
# Display  records from Silver tables

print("===== Silver PdM_errors =====")
display(silver_errors.limit(20))

print("===== Silver PdM_failures =====")
display(silver_failures.limit(20))

print("===== Silver PdM_machines =====")
display(silver_machines.limit(20))

print("===== Silver PdM_maint =====")
display(silver_maint.limit(20))

print("===== Silver PdM_telemetry =====")
display(silver_telemetry.limit(20))

===== Silver PdM_errors =====


datetime,machineID,errorID
2015-01-03T07:00:00.000Z,1,error1
2015-01-03T20:00:00.000Z,1,error3
2015-01-04T06:00:00.000Z,1,error5
2015-01-10T15:00:00.000Z,1,error4
2015-01-22T10:00:00.000Z,1,error4
2015-01-25T15:00:00.000Z,1,error4
2015-01-27T04:00:00.000Z,1,error1
2015-03-03T22:00:00.000Z,1,error2
2015-03-05T06:00:00.000Z,1,error1
2015-03-20T18:00:00.000Z,1,error1


===== Silver PdM_failures =====


datetime,machineID,failure
2015-01-05T06:00:00.000Z,1,comp4
2015-03-06T06:00:00.000Z,1,comp1
2015-04-20T06:00:00.000Z,1,comp2
2015-06-19T06:00:00.000Z,1,comp4
2015-09-02T06:00:00.000Z,1,comp4
2015-10-17T06:00:00.000Z,1,comp2
2015-12-16T06:00:00.000Z,1,comp4
2015-03-19T06:00:00.000Z,2,comp1
2015-03-19T06:00:00.000Z,2,comp2
2015-04-18T06:00:00.000Z,2,comp2


===== Silver PdM_machines =====


machineID,model,age
1,model3,18
2,model4,7
3,model3,8
4,model3,7
5,model3,2
6,model3,7
7,model3,20
8,model3,16
9,model4,7
10,model3,10


===== Silver PdM_maint =====


datetime,machineID,comp
2014-06-01T06:00:00.000Z,1,comp2
2014-07-16T06:00:00.000Z,1,comp4
2014-07-31T06:00:00.000Z,1,comp3
2014-12-13T06:00:00.000Z,1,comp1
2015-01-05T06:00:00.000Z,1,comp4
2015-01-05T06:00:00.000Z,1,comp1
2015-01-20T06:00:00.000Z,1,comp3
2015-01-20T06:00:00.000Z,1,comp1
2015-02-04T06:00:00.000Z,1,comp4
2015-02-04T06:00:00.000Z,1,comp3


===== Silver PdM_telemetry =====


datetime,machineID,volt,rotate,pressure,vibration
2015-01-01T16:00:00.000Z,1,167.060980719256,382.483542906686,103.780662505568,42.6757999060571
2015-01-03T02:00:00.000Z,1,177.70644167607,521.837936384375,101.528033099946,35.4684440422458
2015-01-03T17:00:00.000Z,1,164.148094555294,429.822143515459,104.532123632465,43.7886815487723
2015-01-04T02:00:00.000Z,1,165.187401310127,475.126936759955,114.584420977964,57.5490157337645
2015-01-04T11:00:00.000Z,1,215.656488291429,458.097746169369,95.0362798574919,51.6479808086499
2015-01-05T07:00:00.000Z,1,155.511451847996,498.398434757957,103.068134356782,33.2704150617335
2015-01-08T00:00:00.000Z,1,172.379381592369,528.596952137017,105.052661897191,36.865654004127
2015-01-10T01:00:00.000Z,1,176.654973555311,416.429099834853,108.09370833812,45.8347350214457
2015-01-10T12:00:00.000Z,1,162.433227131287,511.288171366559,119.749926388903,36.85942367168
2015-01-10T15:00:00.000Z,1,159.251349513939,462.691198384678,108.232985430568,31.2000279004032


In [0]:
# Compare Bronze and Silver row counts

tables = [
    ("PdM_errors", f"{bronze_path}/PdM_errors", f"{silver_path}/PdM_errors"),
    ("PdM_failures", f"{bronze_path}/PdM_failures", f"{silver_path}/PdM_failures"),
    ("PdM_machines", f"{bronze_path}/PdM_machines", f"{silver_path}/PdM_machines"),
    ("PdM_maint", f"{bronze_path}/PdM_maint", f"{silver_path}/PdM_maint"),
    ("PdM_telemetry", f"{bronze_path}/PdM_telemetry", f"{silver_path}/PdM_telemetry")
]

for table_name, bronze_table, silver_table in tables:
    bronze_count = spark.read.format("delta").load(bronze_table).count()
    silver_count = spark.read.format("delta").load(silver_table).count()
    
    status = "PASS" if bronze_count == silver_count else "CHECK"
    
    print(f"{table_name}: Bronze = {bronze_count}, Silver = {silver_count}, Status = {status}")

PdM_errors: Bronze = 3919, Silver = 3919, Status = PASS
PdM_failures: Bronze = 761, Silver = 761, Status = PASS
PdM_machines: Bronze = 100, Silver = 100, Status = PASS
PdM_maint: Bronze = 3286, Silver = 3286, Status = PASS
PdM_telemetry: Bronze = 876100, Silver = 876100, Status = PASS
